In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import dotenv
import pydantic
import dspy

In [ ]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.pardir, os.path.pardir, os.path.pardir))
DOTENV_FILE = os.path.join(PROJECT_ROOT, '.env')


dotenv.load_dotenv(DOTENV_FILE)

In [ ]:
lm = dspy.LM('openai/gpt-4o-mini')
dspy.configure(lm=lm)

# Tools

DSPy provides powerful support for **tool-using agents** that can interact with external functions, APIs, and services. Tools enable language models to go beyond text generation by performing actions, retrieving information, and processing data dynamically.

There are two main approaches to using tools in DSPy:

1. **`dspy.ReAct`** - A fully managed tool agent that handles reasoning and tool calls automatically
2. **Manual tool handling** - Direct control over tool calls using `dspy.Tool`, `dspy.ToolCalls`, and custom signatures

## Approach 1: Using `dspy.ReAct` (Fully Managed)

The `dspy.ReAct` module implements the Reasoning and Acting (ReAct) pattern, where the language model iteratively reasons about the current situation and decides which tools to call.

### Basic Example

In [ ]:
# Define your tools as functions
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # In a real implementation, this would call a weather API
    return f"The weather in {city} is sunny and 75°F"

def search_web(query: str) -> str:
    """Search the web for information."""
    # In a real implementation, this would call a search API
    return f"Search results for '{query}': [relevant information...]"

# Create a ReAct agent
react_agent = dspy.ReAct(
    signature="question -> answer",
    tools=[get_weather, search_web],
    max_iters=5
)

# Use the agent
result = react_agent(question="What's the weather like in Tokyo?")
print(result.answer)
print("Tool calls made:", result.trajectory)

### ReAct Features
- **Automatic reasoning**: The model thinks through the problem step by step
- **Tool selection**: Automatically chooses which tool to use based on the situation
- **Iterative execution**: Can make multiple tool calls to gather information
- **Error handling**: Built-in error recovery for failed tool calls
- **Trajectory tracking**: Complete history of reasoning and tool calls

### ReAct Parameters

In [ ]:
react_agent = dspy.ReAct(
    signature="question -> answer",  # Input/output specification
    tools=[tool1, tool2, tool3],     # List of available tools
    max_iters=10                     # Maximum number of tool call iterations
)

## Approach 2: Manual Tool Handling
For more control over the tool calling process, you can manually handle tools using DSPy's tool types.

### Basic Setup

In [ ]:

class ToolSignature(dspy.Signature):
    """Signature for manual tool handling."""
    question: str = dspy.InputField()
    tools: list[dspy.Tool] = dspy.InputField()
    outputs: dspy.ToolCalls = dspy.OutputField()

def weather(city: str) -> str:
    """Get weather information for a city."""
    return f"The weather in {city} is sunny"

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = eval(expression)  # Note: Use safely in production
        return f"The result is {result}"
    except:
        return "Invalid expression"

# Create tool instances
tools = {
    "weather": dspy.Tool(weather),
    "calculator": dspy.Tool(calculator)
}

# Create predictor
predictor = dspy.Predict(ToolSignature)

# Make a prediction
response = predictor(
    question="What's the weather in New York?",
    tools=list(tools.values())
)

# Execute the tool calls
for call in response.outputs.tool_calls:
    # Execute the tool call
    result = tools[call.name](**call.args)
    print(f"Tool: {call.name}")
    print(f"Args: {call.args}")
    print(f"Result: {result}")

# In the future above 3.0.3
# for call in response.outputs.tool_calls:
#     # Execute the tool call
#     result = call.execute()k
#     print(f"Tool: {call.name}")
#     print(f"Args: {call.args}")
#     print(f"Result: {result}")

## Understanding `dspy.Tool`
The `dspy.Tool` class wraps regular Python functions to make them compatible with DSPy's tool system:

In [ ]:
def my_function(param1: str, param2: int = 5) -> str:
    """A sample function with parameters."""
    return f"Processed {param1} with value {param2}"

# Create a tool
tool = dspy.Tool(my_function)

# Tool properties
print(tool.name)        # "my_function"
print(tool.desc)        # The function's docstring
print(tool.args)        # Parameter schema
print(str(tool))        # Full tool description

## Understanding `dspy.ToolCalls`
The `dspy.ToolCalls` type represents the output from a model that can make tool calls. Each individual tool call can be executed using the `execute` method:

In [ ]:
# After getting a response with tool calls
for call in response.outputs.tool_calls:
    print(f"Tool name: {call.name}")
    print(f"Arguments: {call.args}")

    # Execute individual tool calls with different options:

    # Option 1: Automatic discovery (finds functions in locals/globals)
    result = call.execute()  # Automatically finds functions by name

    # Option 2: Pass tools as a dict (most explicit)
    result = call.execute(functions={"weather": weather, "calculator": calculator})

    # Option 3: Pass Tool objects as a list
    result = call.execute(functions=[dspy.Tool(weather), dspy.Tool(calculator)])

    print(f"Result: {result}")

## Using Native Tool Calling
DSPy adapters support **native function calling**, which leverages the underlying language model's built-in tool calling capabilities rather than relying on text-based parsing. This approach can provide more reliable tool execution and better integration with models that support native function calling.

!! **NOTE**: Native tool calling doesn't guarantee better quality. It's possible that native tool calling produces lower quality than custom tool calling.

### Adapter Behavior
Different DSPy adapters have different defaults for native function calling:

- **ChatAdapter** - Uses `use_native_function_calling=False` by default (relies on text parsing)
- **JSONAdapter** - Uses `use_native_function_calling=True` by default (uses native function calling)

You can override these defaults by explicitly setting the `use_native_function_calling` parameter when creating an adapter.

### Configuration

In [ ]:
# ChatAdapter with native function calling enabled
chat_adapter_native = dspy.ChatAdapter(use_native_function_calling=True)

# JSONAdapter with native function calling disabled
json_adapter_manual = dspy.JSONAdapter(use_native_function_calling=False)

# Configure DSPy to use the adapter
dspy.configure(lm=dspy.LM(model="openai/gpt-4o"), adapter=chat_adapter_native)

You can enable the [MLflow tracing](https://dspy.ai/tutorials/observability/) to check how native tool calling is being used. If you use `JSONAdapter` or `ChatAdapter` with native function calling enabled on the code snippet as provided in [the section above](https://dspy.ai/learn/programming/tools/#basic-setup), you should see native function calling arg `tools` is set like the screenshot below:

native tool calling

### Model Compatibility
Native function calling automatically detects model support using `litellm.supports_function_calling()`. If the model doesn't support native function calling, DSPy will fall back to manual text-based parsing even when `use_native_function_calling=True` is set.

## Best Practices

### 1. Tool Function Design
- **Clear docstrings**: Tools work better with descriptive documentation
- **Type hints**: Provide clear parameter and return types
- **Simple parameters**: Use basic types (str, int, bool, dict, list) or Pydantic models

In [ ]:
def good_tool(city: str, units: str = "celsius") -> str:
    """
    Get weather information for a specific city.

    Args:
        city: The name of the city to get weather for
        units: Temperature units, either 'celsius' or 'fahrenheit'

    Returns:
        A string describing the current weather conditions
    """
    # Implementation with proper error handling
    if not city.strip():
        return "Error: City name cannot be empty"

    # Weather logic here...
    return f"Weather in {city}: 25°{units[0].upper()}, sunny"

### 2. Choosing Between ReAct and Manual Handling¶

**Use `dspy.ReAct` when:**

- You want automatic reasoning and tool selection
- The task requires multiple tool calls
- You need built-in error recovery
- You want to focus on tool implementation rather than orchestration

**Use manual tool handling when:**

- You need precise control over tool execution
- You want custom error handling logic
- You want to minimize the latency
- Your tool returns nothing (void function)

Tools in DSPy provide a powerful way to extend language model capabilities beyond text generation. Whether using the fully automated ReAct approach or manual tool handling, you can build sophisticated agents that interact with the world through code.